In [1]:
# import libraries 
import time
import re
import datetime
import csv
import logging
import sys
import os
import random

from selenium import webdriver
from bs4 import BeautifulSoup

# cần 2 cái dưới để điều hướng con trỏ chuột sang trang kế
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import logging # dùng để ghi tiến trình & lỗi khi crawl data

# Import 3 vũ khí mới để vượt qua bước Login của web
import os # để đọc thông tin trong file `.env`
import undetected_chromedriver as uc
from dotenv import load_dotenv

In [2]:
# --- Thiết lập logging ---
logging.basicConfig(
    filename="logging_topcv.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding='utf-8'
)

# 1. Truy cập web

In [3]:
# https://www.topcv.vn/tim-viec-lam-data-analyst?sort=new&type_keyword=1&page=2

KEYWORDS = ["Data Analyst"] # Rút gọn để test
BASE_URL_TEMPLATE = "https://www.topcv.vn/tim-viec-lam-{keyword}?sort=new&type_keyword=1&page={page}"
MAX_PAGES = 3
OUTPUT_CSV = "data_topcv.csv"

# ── 3. LOGGING (Giữ nguyên của bạn) ──────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

In [4]:
# THAY ĐỔI 1: Dùng undetected_chromedriver thay vì webdriver.Chrome()
options = uc.ChromeOptions()


# Thêm cấu hình vô hiệu hóa chức năng hỏi lưu mật khẩu VÀ chặn thông báo (Notifications)
prefs = {
    "credentials_enable_service": False,
    "profile.password_manager_enabled": False,
    "profile.default_content_setting_values.notifications": 2  # Thêm dòng này! 2 có nghĩa là Block

}
options.add_experimental_option("prefs", prefs)



# THÊM version_main=140 VÀO ĐÂY vì nó mặc định chạy ver Chrome 147
driver = uc.Chrome(options=options, version_main=140) 


# Thay vì maximize_window(), dùng set_window_size và set_window_position
# Đặt kích thước cho trình duyệt (chiều rộng x chiều cao)
driver.set_window_size(900, 1080) # Bạn có thể thay đổi số theo độ phân giải màn hình
# (Tùy chọn) Đặt vị trí xuất hiện của cửa sổ (x, y) trên màn hình (0,0 là góc trên cùng bên trái)
driver.set_window_position(900, 0) # Có thể đẩy nó sang lề phải của màn hình

{'height': 882, 'width': 903, 'x': 900, 'y': 0}

# 4. Bắt đầu lấy data sản phẩm nè.
Dưới đây là các thông tin của 1 job có link bình thường

1.	`job_url`: Link chi tiết của job đó.
2.	`job_title`: Tiêu đề công việc.

3.	`salary`: Mức lương (thỏa thuận OR rank min - max).
4.	`location`: Địa điểm làm việc.
5.	`years_experience`: kinh nghiệm

6.	`job_deadline`: (Hạn nộp hồ sơ)
7.	`date_post`: Tính từ ngày dealine trừ đi 1 tháng

8.	`company_name`: Tên công ty.
9.	`company_industry`: Lĩnh vực cty (Dược phẩm / Y tế / Công nghệ sinh học)

10.	`level`: cấp bậc (Nhân viên)
11.	`education_level`: (Trình độ học vấn) (cao đẳng trở lên)
12.	`hire_number`: Số lượng tuyển
13.	`job_type`: Hình thức làm việc

14.	`expertise`: Chuyên môn (Data Analyst Chăm sóc sức khỏe / Làm đẹp Y tế / Dược phẩm)

15.	`job_description`: Mô tả công việc.
16.	`requirements`: Yêu cầu công việc (Kinh nghiệm, kỹ năng).
17.	`benefits`: Phúc lợi (Lý do tại sao bạn sẽ yêu thích làm việc tại đây).



16. `date_post`
```py
Về date_post (Ngày đăng). Có lẽ t sẽ cào nó bên ngoài, vì nó chỉ ở bên ngoài
1. lấy list link trong 1 page đính kèm date post ( NOW - số ngày đã up)
2. Check duplicate link
3. Đi vào crawl chi tiết list link
4. Next trang tiếp theo
```

## Nhóm 1: Trang ko VIP
Là job có `company_tag = soup.find('h1', class_="title")` "vì đây là tag duy nhất"

- https://www.topcv.vn/brand/giaoducsec/tuyen-dung/chuyen-vien-chuyen-doi-so-lam-viec-tai-dong-da-thu-nhap-hap-dan-j2108055.html
- https://www.topcv.vn/brand/galaxydebt/tuyen-dung/chuyen-vien-phan-tich-du-lieu-nganh-tai-chinh-j2104686.html
- https://www.topcv.vn/brand/scotsenglishaustralia/tuyen-dung/chuyen-vien-quan-ly-du-lieu-van-hanh-luong-upto-15-trieu-j2103483.html
- https://www.topcv.vn/brand/oceaneduvn/tuyen-dung/chuyen-vien-du-lieu-j2091143.html



In [5]:
# Link web đặc biệt
# https://www.topcv.vn/brand/giaoducsec/tuyen-dung/chuyen-vien-chuyen-doi-so-lam-viec-tai-dong-da-thu-nhap-hap-dan-j2108055.html

job_link = 'https://www.topcv.vn/brand/giaoducsec/tuyen-dung/chuyen-vien-chuyen-doi-so-lam-viec-tai-dong-da-thu-nhap-hap-dan-j2108055.html'
job_link = job_link.split('?')[0]
job_link

'https://www.topcv.vn/brand/giaoducsec/tuyen-dung/chuyen-vien-chuyen-doi-so-lam-viec-tai-dong-da-thu-nhap-hap-dan-j2108055.html'

In [6]:

# Mở trang job chi tiết
driver.get(job_link)
time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn

# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [7]:
# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

In [8]:
soup = new_soup

### 1. Title

In [14]:
# (?=.*A)(?=.*B) nghĩa là: Quét xem có chứa A không, VÀ quét xem có chứa B không
pattern = re.compile(r'(?=.*premium-job-basic-information)(?=.*title)', re.IGNORECASE)

title_tag = soup.find('h2', class_=pattern)
title_tag

<h2 class="premium-job-basic-information__content--title" style="">
                            Chuyên Viên Chuyển Đổi Số Làm Việc Tại <a class="text-highlight" href="https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-dong-da-l1d104" target="_blank" title="Việc làm tại Đống Đa">Đống Đa</a> - Thu Nhập Hấp Dẫn
                                    <span class="icon-verified-employer level-five">
<i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
  &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã được duyệt Giấy phép kinh doanh&lt;/span&gt;&

In [16]:
title = title_tag.get_text(separator=' ', strip=True)
title

'Chuyên Viên Chuyển Đổi Số Làm Việc Tại Đống Đa - Thu Nhập Hấp Dẫn'

In [9]:
# function 1: lấy tiêu đề job

def get_title(soup):
    try:
        # 1. (?=.*A)(?=.*B) nghĩa là: Quét xem có chứa A không, VÀ quét xem có chứa B không
        pattern = re.compile(r'(?=.*premium-job-basic-information)(?=.*title)', re.IGNORECASE)
        
        if pattern:
            # 2. Tìm thẻ <h1> duy nhất bên trong cái hộp đó
            # Không cần quan tâm class của nó là gì
            title_tag = soup.find('h2', class_=pattern)

            
            if title_tag:
                # 3. Lấy text và làm sạch
                # Lưu ý: Thẻ h1 này có chứa một cái icon (thẻ span class="icon-verified-employer")
                # Nếu bạn dùng get_text() bình thường, nó có thể lôi cả text ẩn trong icon ra.
                # Để an toàn, ta chỉ lấy text của thẻ <a> và các text node trực tiếp của <h1>
                
                # Cách an toàn nhất: Lấy toàn bộ text, sau đó xóa các khoảng trắng thừa
                raw_title = title_tag.get_text(separator=' ', strip=True)
                
                # (Tùy chọn) Nếu cái icon có chứa text rác, bạn có thể xóa thẻ span đó đi trước khi lấy text
                # icon_span = title_tag.find('span', class_='icon-verified-employer')
                # if icon_span:
                #     icon_span.decompose() # Xóa thẻ span khỏi cây HTML
                # return title_tag.get_text(strip=True)
                
                return raw_title
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy title: {e}")
        return None

In [10]:
# test function: lấy tiêu đề job
title = get_title(new_soup)
title

'Chuyên Viên Chuyển Đổi Số Làm Việc Tại Đống Đa - Thu Nhập Hấp Dẫn'

### 2. Salary
### 3. location
### 4. years_experience  

Vì 3 mục ở chung phần HEAD_BOX --> Cào kiểu `Dictionary`

In [20]:
# 2. Lấy thẻ cha của title là Head_box
head_box = title_tag.parent
head_box

<div class="premium-job-basic-information__content">
<h2 class="premium-job-basic-information__content--title" style="">
                            Chuyên Viên Chuyển Đổi Số Làm Việc Tại <a class="text-highlight" href="https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-dong-da-l1d104" target="_blank" title="Việc làm tại Đống Đa">Đống Đa</a> - Thu Nhập Hấp Dẫn
                                    <span class="icon-verified-employer level-five">
<i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
  &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i

In [28]:
# Big_group  là cái khung to chứa 3 cái ô nhỏ kia
# premium-job-basic-information__content--sections

# 1. (?=.*A)(?=.*B) nghĩa là: Quét xem có chứa A không, VÀ quét xem có chứa B không
pattern_big_groups = re.compile(r'(?=.*premium-job-basic-information)(?=.*sections)', re.IGNORECASE)
        
if pattern_big_groups:
    big_group = head_box.find('div', class_=pattern_big_groups)

big_group

<div class="premium-job-basic-information__content--sections">
<div class="basic-information-item">
<div class="basic-information-item__logo" data-original-title="Thu nhập khi đạt 100% KPI" data-toggle="tooltip" title="">
<svg fill="none" height="24" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg">
<path d="M21.92 16.75C21.59 19.41 19.41 21.59 16.75 21.92C15.14 22.12 13.64 21.68 12.47 20.82C11.8 20.33 11.96 19.29 12.76 19.05C15.77 18.14 18.14 15.76 19.06 12.75C19.3 11.96 20.34 11.8 20.83 12.46C21.68 13.64 22.12 15.14 21.92 16.75Z" fill="white"></path>
<path d="M9.99 2C5.58 2 2 5.58 2 9.99C2 14.4 5.58 17.98 9.99 17.98C14.4 17.98 17.98 14.4 17.98 9.99C17.97 5.58 14.4 2 9.99 2ZM9.05 8.87L11.46 9.71C12.33 10.02 12.75 10.63 12.75 11.57C12.75 12.65 11.89 13.54 10.84 13.54H10.75V13.59C10.75 14 10.41 14.34 10 14.34C9.59 14.34 9.25 14 9.25 13.59V13.53C8.14 13.48 7.25 12.55 7.25 11.39C7.25 10.98 7.59 10.64 8 10.64C8.41 10.64 8.75 10.98 8.75 11.39C8.75 11.75 9.01 12.04 9.33 12.0

In [27]:
# Tìm 3 cái khung nhỏ, bao từng cặp từng cặp giá trị
# basic-information-item__data

groups = big_group.find_all('div', class_='basic-information-item__data')
print(len(groups), " cặp giá trị")
groups

3  cặp giá trị


[<div class="basic-information-item__data">
 <div class="basic-information-item__data--label">Mức lương</div>
 <div class="basic-information-item__data--value">Thoả thuận</div>
 </div>,
 <div class="basic-information-item__data">
 <div class="basic-information-item__data--label">Địa điểm</div>
 <div class="basic-information-item__data--value">
 <div class="job-detail__info--section-content-value text-location"><span class="text-highlight" data-html="true" data-original-title="
             &lt;div style='text-align: left; display: flex; flex-direction: column;'&gt;
                 &lt;p style='color: #4FF398; margin: 0 0 4px 0; font-weight: 400; font-size: 12px; line-height: 18px; letter-spacing: 0.12px;'&gt;
                     Địa điểm làm việc đã được cập nhật theo Danh mục Hành chính mới (thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)
                 &lt;/p&gt;
                 &lt;ul style='margin: 0; padding-left: 24px; font-weight: 400; font-size: 12px; line-height: 18px; l

In [ ]:
# Dictionary: LƯƠNG + VỊ TRÍ + NĂM KINH NGHIỆM


def job_head_info(soup):
    """
    Quét HTML 1 lần, gom toàn bộ thông tin thành Dictionary
    """
    job_head_info = {}
    
    try:
        # 1. Tìm khối lớn chứa thông tin

        # 1.1. Lấy tab title
        pattern = re.compile(r'(?=.*premium-job-basic-information)(?=.*title)', re.IGNORECASE)
        title_tag = soup.find('h2', class_=pattern)

        # 1.2. Lấy thẻ cha của title là Head_box
        head_box = title_tag.parent

        # 1.3. Lấy Big_group là cái khung to chứa 3 cái ô nhỏ
        pattern_big_groups = re.compile(r'(?=.*premium-job-basic-information)(?=.*sections)', re.IGNORECASE)
        
        if pattern_big_groups:
            big_group = head_box.find('div', class_=pattern_big_groups)


        if big_group:
            # Tìm tất cả các khối nhỏ chứa thông tin
            groups = big_group.find_all('div', class_='basic-information-item__data')
            
            
            for group in groups:
                pattern_lable = re.compile(r'(?=.*basic-information-item)(?=.*label)', re.IGNORECASE)
                pattern_value = re.compile(r'(?=.*basic-information-item)(?=.*value)', re.IGNORECASE)

                lable_tag = group.find('div', class_= pattern_lable)
                value_tag = group.find('div', class_= pattern_value)
                
                # Chỉ lưu nếu có đủ cả title và value
                if lable_tag and value_tag:
                    lable_text = lable_tag.get_text(separator=" ", strip=True)
                    value_text = value_tag.get_text(separator=" ", strip=True)
                    job_head_info[lable_text] = value_text
                    
        return job_head_info

            
    except Exception as e:
        logging.error(f"Lỗi khi lấy general_info: {e}")
        return job_head_info # Trả về dict rỗng nếu lỗi

# ================= ÁP DỤNG =================

# Gọi hàm 1 lần duy nhất
job_head_info = job_head_info(soup)
job_head_info


{'Mức lương': 'Thoả thuận', 'Địa điểm': 'Hà Nội', 'Kinh nghiệm': '2 năm'}

In [30]:
# Lấy từng giá trị ra cực kỳ nhanh và gọn bằng hàm .get() của Dict
# Nếu ko có, thì .get() trả về NONE
salary              = job_head_info.get('Mức lương')
location            = job_head_info.get('Địa điểm')
years_experience    = job_head_info.get('Kinh nghiệm')

In [31]:
salary            

'Thoả thuận'

In [32]:
location          

'Hà Nội'

In [33]:
years_experience  

'2 năm'

Cả 4 cái này cùng nằm trong 1 Khối => Dùng `Dictionary`


### 5.  level_employee
cấp bậc (Nhân viên)

### 6.  education_level
### 7.  hire_number
### 8.  work_type
Hình thức làm việc (Toàn thời gian)
### 9.  job_deadline

In [24]:
# 1. (?=.*A)(?=.*B) nghĩa là: Quét xem có chứa A không, VÀ quét xem có chứa B không
pattern = re.compile(r'(?=.*premium-job-general-information)(?=.*title)', re.IGNORECASE)

if pattern :
    # 2. Tìm thẻ <h2> "Thông tin chung" duy nhất bên trong cái hộp đó
    title_box_tag = soup.find(
        'h2', 
        class_=pattern, 
        string=re.compile(r'Thông tin chung', re.IGNORECASE)
        )
    
title_box_tag

<h2 class="premium-job-general-information__title">
        Thông tin chung
    </h2>

In [26]:
# Tìm khối lớn chứa thông tin
general_content_box = title_box_tag.parent
general_content_box

<div class="premium-job-general-information">
<h2 class="premium-job-general-information__title">
        Thông tin chung
    </h2>
<div class="premium-job-general-information__content">
<div class="premium-job-general-information__content--row">
<div class="general-information-logo">
<i class="fa-solid fa-chart-user"></i>
</div>
<div class="general-information-data">
<div class="general-information-data__label">Cấp bậc</div>
<div class="general-information-data__value">Nhân viên</div>
</div>
</div>
<div class="premium-job-general-information__content--row">
<div class="general-information-logo">
<svg height="22" viewbox="0 0 17 18" width="22" xmlns="http://www.w3.org/2000/svg">
<path d="M11.8812 11.548C12.3502 11.24 12.9662 11.576 12.9662 12.136V13.039C12.9662 13.928 12.2732 14.88 11.4402 15.16L9.20718 15.902C8.81518 16.035 8.17818 16.035 7.79318 15.902L5.56018 15.16C4.72018 14.88 4.03418 13.928 4.03418 13.039V12.129C4.03418 11.576 4.65018 11.24 5.11218 11.541L6.55418 12.479C7.10718 1

In [ ]:
# Tìm tất cả các khối chứa thông tin (label + value)
info_blocks = general_content_box.find_all('div', class_="general-information-data")
info_blocks

[<div class="general-information-data">
 <div class="general-information-data__label">Cấp bậc</div>
 <div class="general-information-data__value">Nhân viên</div>
 </div>,
 <div class="general-information-data">
 <div class="general-information-data__label">Học vấn</div>
 <div class="general-information-data__value">Đại Học trở lên</div>
 </div>,
 <div class="general-information-data">
 <div class="general-information-data__label">Số lượng tuyển</div>
 <div class="general-information-data__value">4 người</div>
 </div>,
 <div class="general-information-data">
 <div class="general-information-data__label">Hình thức làm việc</div>
 <div class="general-information-data__value">Toàn thời gian</div>
 </div>,
 <div class="general-information-data">
 <div class="general-information-data__label">Hạn nộp hồ sơ</div>
 <div class="general-information-data__value">
                                             03/05/2026
                                     </div>
 </div>]

In [31]:
job_general_info = {}

for block in info_blocks:
    pattern_lable = re.compile(r'(?=.*general-information-data)(?=.*label)', re.IGNORECASE)
    pattern_value = re.compile(r'(?=.*general-information-data)(?=.*value)', re.IGNORECASE)

    lable_tag = block.find('div', class_= pattern_lable)
    value_tag = block.find('div', class_= pattern_value)
    
    # Chỉ lưu nếu có đủ cả title và value
    if lable_tag and value_tag:
        lable_text = lable_tag.get_text(separator=" ", strip=True)
        value_text = value_tag.get_text(separator=" ", strip=True)
        job_general_info[lable_text] = value_text

# Xem toàn bộ dữ liệu cào được
print(job_general_info)

{'Cấp bậc': 'Nhân viên', 'Học vấn': 'Đại Học trở lên', 'Số lượng tuyển': '4 người', 'Hình thức làm việc': 'Toàn thời gian', 'Hạn nộp hồ sơ': '03/05/2026'}


In [32]:
# Dictionary: CẤP BẬC + HỌC VẤN + HIRE_NUMBER + WORK_TYPE + DATE_DEADLINE


def job_general_info(soup):
    """
    Quét HTML 1 lần, gom toàn bộ thông tin thành Dictionary
    """
    job_general_info = {}
    
    try:
        # 1. Tìm khối lớn chứa thông tin

        # 1.1. Lấy tab title
        pattern = re.compile(r'(?=.*premium-job-general-information)(?=.*title)', re.IGNORECASE)

        if pattern :
            # Tìm thẻ <h2> "Thông tin chung" duy nhất bên trong cái hộp đó
            title_box_tag = soup.find(
                'h2', 
                class_=pattern, 
                string=re.compile(r'Thông tin chung', re.IGNORECASE)
                )

        # 1.2. Lấy thẻ cha của tab_title là general_content_box
        general_content_box = title_box_tag.parent



        if general_content_box:
            # Tìm tất cả các khối nhỏ chứa thông tin (label + value)
            info_blocks = general_content_box.find_all('div', class_="general-information-data")
            
            
            for block in info_blocks:
                pattern_lable = re.compile(r'(?=.*general-information-data)(?=.*label)', re.IGNORECASE)
                pattern_value = re.compile(r'(?=.*general-information-data)(?=.*value)', re.IGNORECASE)

                lable_tag = block.find('div', class_= pattern_lable)
                value_tag = block.find('div', class_= pattern_value)
                
                # Chỉ lưu nếu có đủ cả title và value
                if lable_tag and value_tag:
                    lable_text = lable_tag.get_text(separator=" ", strip=True)
                    value_text = value_tag.get_text(separator=" ", strip=True)
                    job_general_info[lable_text] = value_text
                    
        return job_general_info

            
    except Exception as e:
        logging.error(f"Lỗi khi lấy job_general_info: {e}")
        return job_head_info # Trả về dict rỗng nếu lỗi

# ================= ÁP DỤNG =================

# Gọi hàm 1 lần duy nhất
job_general_info = job_general_info(soup)
job_general_info


{'Cấp bậc': 'Nhân viên',
 'Học vấn': 'Đại Học trở lên',
 'Số lượng tuyển': '4 người',
 'Hình thức làm việc': 'Toàn thời gian',
 'Hạn nộp hồ sơ': '03/05/2026'}

In [33]:
# Lấy từng giá trị ra cực kỳ nhanh và gọn bằng hàm .get() của Dict
# Nếu ko có, thì .get() trả về NONE
level_employee  = job_general_info.get('Cấp bậc')
education_level = job_general_info.get('Học vấn')
hire_number     = job_general_info.get('Số lượng tuyển')
work_type       = job_general_info.get('Hình thức làm việc')
job_deadline    = job_general_info.get('Hạn nộp hồ sơ')

In [34]:
level_employee

'Nhân viên'

In [36]:
education_level

'Đại Học trở lên'

In [35]:
hire_number 

'4 người'

In [37]:
job_deadline

'03/05/2026'

### 10.     company_name

In [ ]:

# 1. Tìm thẻ <h1> "title" duy nhất chứa tên công ty
company_tag = soup.find('h1', class_="title")
company_tag

<h1 class="title">CÔNG TY CỔ PHẦN ĐẦU TƯ VÀ GIÁO DỤC QUỐC TẾ SEC</h1>

In [41]:
company_name = company_tag.get_text(strip=True)
company_name

'CÔNG TY CỔ PHẦN ĐẦU TƯ VÀ GIÁO DỤC QUỐC TẾ SEC'

In [42]:
# function 6: lấy tên cty

def get_company(soup):
    try:
        # 2. Tìm thẻ <h1> "title" duy nhất chứa tên công ty
        company_tag = soup.find('h1', class_="title")
        
        if company_tag:
            return company_tag.get_text(strip=True)

        return None
    except Exception:
        return None


company_name = get_company(soup)
company_name

'CÔNG TY CỔ PHẦN ĐẦU TƯ VÀ GIÁO DỤC QUỐC TẾ SEC'

### 11.     expertise
Chuyên môn của nhân viên (Data Analyst Chăm sóc sức khỏe / Làm đẹp Y tế / Dược phẩm)

In [13]:
# 1. Xác định box chứa phần chuyên môn của nhân viên "job-tags__group-name"
pattern_class = re.compile(r'(?=.*job-tags)(?=.*group-name)', re.IGNORECASE)
pattern_string=re.compile(r'Chuyên môn', re.IGNORECASE)

expertise_tag = soup.find('div', class_=pattern_class, string=pattern_string)
expertise_tag

<div class="job-tags__group-name">
                    Chuyên môn:
                </div>

In [14]:
if expertise_tag:
    # 2. Lấy thẻ cha của nó
    parent_expertise = expertise_tag.parent 

parent_expertise

<div class="job-tags__group">
<div class="job-tags__group-name">
                    Chuyên môn:
                </div>
<div class="job-tags__group-list-tag">
<div class="job-tags__group-list-tag-scroll">
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-data-analyst-cr92cb98cl145" target="_blank" title="Chuyên môn Data Analyst">
                            Chuyên môn Data Analyst
                        </a>
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-moi-nhat?domain_knowledge=3" target="_blank" title="IT - Phần mềm">
                            IT - Phần mềm
                        </a>
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-moi-nhat?domain_knowledge=35" target="_blank" title="Marketing / Quảng cáo">
                            Marketing / Quảng cáo
                        </a>
</div>
</div>
</div>

In [15]:
# Tìm thẻ div anh em tiếp theo có class là 'job-tags__group-list-tag-scroll'
expertise_list = parent_expertise.find('div', class_=re.compile(r'group-list-tag-scroll'))
expertise_list

<div class="job-tags__group-list-tag-scroll">
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-data-analyst-cr92cb98cl145" target="_blank" title="Chuyên môn Data Analyst">
                            Chuyên môn Data Analyst
                        </a>
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-moi-nhat?domain_knowledge=3" target="_blank" title="IT - Phần mềm">
                            IT - Phần mềm
                        </a>
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-moi-nhat?domain_knowledge=35" target="_blank" title="Marketing / Quảng cáo">
                            Marketing / Quảng cáo
                        </a>
</div>

In [16]:

if expertise_list:
    # Lấy tất cả thẻ <a> bên trong
    a_tags = expertise_list.find_all('a')
    
    # Dùng list comprehension để lấy text, bỏ khoảng trắng thừa
    tags_list = [a.get_text(separator=" ", strip=True) for a in a_tags if a.text]
    
    # Nối thành chuỗi cách nhau bởi dấu phẩy
    if tags_list:
        expertise = ", ".join(tags_list)


expertise

'Chuyên môn Data Analyst, IT - Phần mềm, Marketing / Quảng cáo'

In [ ]:
# function 9: Lấy Chuyên môn nghiệp vụ của ứng viên


def get_job_expertise(soup):
    """Lấy danh sách các thẻ chuyên môn (Expertise / Tags) của công việc."""
    try:
        # 1. Tìm thẻ label chứa chữ "Chuyên môn"
        pattern_class = re.compile(r'(?=.*job-tags)(?=.*group-name)', re.IGNORECASE)
        pattern_string=re.compile(r'Chuyên môn', re.IGNORECASE)

        expertise_tag = soup.find('div', class_=pattern_class, string=pattern_string)
        
        # 2. Phải lồng các bước tiếp theo vào TRONG lệnh if này để đảm bảo an toàn
        if expertise_tag:
            parent_expertise = expertise_tag.parent 
            
            if parent_expertise:
                # Tìm khối cuộn chứa các thẻ tag
                expertise_list = parent_expertise.find('div', class_=re.compile(r'group-list-tag-scroll'))
                
                if expertise_list:
                    # Lấy tất cả thẻ <a> bên trong
                    a_tags = expertise_list.find_all('a')
                    
                    # Dùng list comprehension để lấy text, loại bỏ các thẻ rỗng
                    tags_list = [a.get_text(strip=True) for a in a_tags if a.get_text(strip=True)]
                    
                    # Nối thành chuỗi cách nhau bởi dấu phẩy, và xóa hết các text "chuyên môn"
                    if tags_list:
                        return ", ".join(tags_list).replace('Chuyên môn', '').strip()
                            
        # Trả về None nếu trang web thiếu một trong các khối HTML trên
        return None 
        
    except Exception as e:
        # Log lỗi giống hệt các function khác trong file của bạn
        logging.error(f"Lỗi khi lấy job_expertise: {e}")
        return None


In [18]:
expertise_employee = get_job_expertise(new_soup)
expertise_employee

'Chuyên môn Data Analyst, IT - Phần mềm, Marketing / Quảng cáo'

### 12      MÔ TẢ CÔNG VIỆC
### 13      YÊU CẦU
### 14      PHÚC LỢI


In [19]:
# 1. thẻ title của box: premium-job-description__box--title
pattern_class = re.compile(r'(?=.*premium-job-description)(?=.*box)(?=.*title)', re.IGNORECASE)
pattern_string=re.compile(r'Mô tả công việc', re.IGNORECASE)

description_tag = soup.find('h2', class_=pattern_class, string=pattern_string)
description_tag

<h2 class="premium-job-description__box--title">Mô tả công việc</h2>

In [20]:
if description_tag:
    # 2. Lấy thẻ cha của nó
    parent_description = description_tag.parent 

parent_description

<div class="premium-job-description__box">
<h2 class="premium-job-description__box--title">Mô tả công việc</h2>
<div class="premium-job-description__box--content">
<p><strong>1. Vận hành tối ưu hệ thống số và triển khai hệ thống mới phù hợp với yêu cầu doanh nghiệp:</strong></p><p>• Quản trị, vận hành và giám sát các hệ thống công nghệ doanh nghiệp đang sử dụng như LMS, CRM, chatbot, form đăng ký, landing page, email/SMS và các công cụ tự động hóa.</p><p>• Tiếp nhận yêu cầu từ các phòng ban, xử lý lỗi hệ thống, chuẩn hóa quy trình sử dụng và hướng dẫn người dùng nội bộ.</p><p>• Thiết lập quy chuẩn nhập liệu, phân quyền người dùng, kiểm tra tính chính xác và đồng bộ của dữ liệu.</p><p>• Theo dõi hiệu quả vận hành của từng nền tảng và đề xuất cải tiến nhằm giảm thao tác thủ công và tăng tốc độ xử lý.</p><p>• Làm việc với vendor/đơn vị phần mềm để mô tả đúng yêu cầu triển khai vận hành các hệ thống mới theo yêu cầu doanh nghiệp. </p><p><strong>2. Tích hợp API và kết nối nền tảng</strong><

In [21]:
# BOX nội dung: premium-job-description__box--content
pattern_content_class = re.compile(r'(?=.*premium-job-description)(?=.*box)(?=.*content)', re.IGNORECASE)

description_content_tag = parent_description.find('div', class_=pattern_content_class)
description_content_tag

<div class="premium-job-description__box--content">
<p><strong>1. Vận hành tối ưu hệ thống số và triển khai hệ thống mới phù hợp với yêu cầu doanh nghiệp:</strong></p><p>• Quản trị, vận hành và giám sát các hệ thống công nghệ doanh nghiệp đang sử dụng như LMS, CRM, chatbot, form đăng ký, landing page, email/SMS và các công cụ tự động hóa.</p><p>• Tiếp nhận yêu cầu từ các phòng ban, xử lý lỗi hệ thống, chuẩn hóa quy trình sử dụng và hướng dẫn người dùng nội bộ.</p><p>• Thiết lập quy chuẩn nhập liệu, phân quyền người dùng, kiểm tra tính chính xác và đồng bộ của dữ liệu.</p><p>• Theo dõi hiệu quả vận hành của từng nền tảng và đề xuất cải tiến nhằm giảm thao tác thủ công và tăng tốc độ xử lý.</p><p>• Làm việc với vendor/đơn vị phần mềm để mô tả đúng yêu cầu triển khai vận hành các hệ thống mới theo yêu cầu doanh nghiệp. </p><p><strong>2. Tích hợp API và kết nối nền tảng</strong></p><p>• Đọc hiểu tài liệu kỹ thuật, API documentation của các nền tảng đang sử dụng.</p><p>• Phối hợp triển khai

In [120]:
if description_content_tag:
    # Lấy text, dùng dấu cách ' ' để các đoạn <p> không bị dính liền vào nhau
    raw_text = description_content_tag.get_text(separator=' ', strip=True)
    
    # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
    clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)
    
    # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
    final_text = ' '.join(clean_text.split())
    
    # Chống lỗi công thức khi xuất ra file CSV/Excel
    if final_text.startswith(('-', '=', '+', '@')):
        final_text = "'" + final_text


final_text

'Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata...) và xây dựng các chỉ số đo lường chất lượng dữ liệu. Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI. Hỗ trợ tuân thủ các quy định về an toàn thông tin, bảo mật dữ liệu, phân quyền truy cập dữ liệu, đảm bảo luật định (Luật Dữ liệu, Luật Bảo vệ Dữ liệu cá nhân, ...). Phối hợp với các đơn vị nghiệp vụ, sản phẩm, IT và DE/BI để đảm bảo dữ liệu được quản trị đúng chuẩn và nhất quán. Tham gia xây dựng tài liệu, hướng dẫn và đào tạo về Data Governance cho các bên liên quan.'

In [35]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def extract_paragraph_content(soup, heading_keyword):
    """
    Hàm đa năng: Tìm thẻ h2 chứa keyword, sau đó lấy toàn bộ text 
    của các thẻ anh em (siblings) nằm dưới nó trong cùng 1 div.
    """
    try:
        # 1. thẻ title của box: premium-job-description__box--title
        pattern_class = re.compile(r'(?=.*premium-job-description)(?=.*box)(?=.*title)', re.IGNORECASE)
        pattern_string = re.compile(rf'{heading_keyword}', re.IGNORECASE)

        h2_tag = soup.find('h2', class_=pattern_class, string=pattern_string)


        # 2. Tìm thẻ cha là BOX chứa cái thông tin đó
        if h2_tag:
            box_tag = h2_tag.parent 

            # 3. BOX nội dung: premium-job-description__box--content
            if box_tag:
                pattern_content_class = re.compile(r'(?=.*premium-job-description)(?=.*box)(?=.*content)', re.IGNORECASE)
                content_tag = box_tag.find('div', class_=pattern_content_class)

                # 4. Lấy nguyên text
                if content_tag:
                    # Lấy text, dùng dấu cách ' ' để các đoạn <p> không bị dính liền vào nhau
                    raw_text = content_tag.get_text(separator=' ', strip=True)
                    
                    # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
                    clean_text = re.sub(r'[•\*●\-\+]', '', raw_text)
                    
                    # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
                    final_text = ' '.join(clean_text.split())
                    
                    # Chống lỗi công thức khi xuất ra file CSV/Excel
                    if final_text.startswith(('-', '=', '+', '@')):
                        final_text = "'" + final_text
                
                    return final_text
            
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {heading_keyword}: {e}")
        return None



# ==========================================
# ÁP DỤNG VÀO 3 HÀM CỦA BẠN
# ==========================================

def get_job_description(soup):
    return extract_paragraph_content(soup, "Mô tả công việc")

def get_job_requirements(soup):
    return extract_paragraph_content(soup, "Yêu cầu ứng viên")

def get_benefits(soup):
    return extract_paragraph_content(soup, "Quyền lợi được hưởng")

In [36]:
description = get_job_description(soup)
description

'1. Vận hành tối ưu hệ thống số và triển khai hệ thống mới phù hợp với yêu cầu doanh nghiệp: Quản trị, vận hành và giám sát các hệ thống công nghệ doanh nghiệp đang sử dụng như LMS, CRM, chatbot, form đăng ký, landing page, email/SMS và các công cụ tự động hóa. Tiếp nhận yêu cầu từ các phòng ban, xử lý lỗi hệ thống, chuẩn hóa quy trình sử dụng và hướng dẫn người dùng nội bộ. Thiết lập quy chuẩn nhập liệu, phân quyền người dùng, kiểm tra tính chính xác và đồng bộ của dữ liệu. Theo dõi hiệu quả vận hành của từng nền tảng và đề xuất cải tiến nhằm giảm thao tác thủ công và tăng tốc độ xử lý. Làm việc với vendor/đơn vị phần mềm để mô tả đúng yêu cầu triển khai vận hành các hệ thống mới theo yêu cầu doanh nghiệp. 2. Tích hợp API và kết nối nền tảng Đọc hiểu tài liệu kỹ thuật, API documentation của các nền tảng đang sử dụng. Phối hợp triển khai kết nối dữ liệu giữa CRM, LMS, landing page, chatbot, công cụ quảng cáo, Google Sheets, dashboard và các phần mềm nội bộ. Thực hiện các tác vụ kỹ thuậ

In [37]:
requirements = get_job_requirements(soup)
requirements

'1. Kiến thức và kinh nghiệm Tốt nghiệp Cao đẳng/Đại học chuyên ngành CNTT, Hệ thống thông tin, Khoa học dữ liệu, Phân tích dữ liệu hoặc các ngành liên quan. Có tối thiểu 2 năm kinh nghiệm ở vị trí tương đương hoặc các vị trí liên quan đến vận hành hệ thống, chuyển đổi số, BA kỹ thuật, data operations hoặc automation. Có kinh nghiệm làm việc với ít nhất một trong các nhóm hệ thống: CRM, LMS, chatbot, landing page, form, automation tools, reporting tools. 2. Kỹ năng kỹ thuật bắt buộc Hiểu nguyên lý hoạt động của API, webhook, request/response, JSON, authentication token. Biết sử dụng Postman hoặc công cụ tương đương để test API. Có thể đọc hiểu và chỉnh sửa code mức cơ bản bằng JavaScript, Python, PHP hoặc ngôn ngữ tương đương. Có khả năng viết câu lệnh SQL cơ bản để truy xuất, kiểm tra hoặc đối soát dữ liệu. Thành thạo Excel/Google Sheets nâng cao: lookup, query, pivot, dashboard cơ bản. Có khả năng sử dụng Power BI, Looker Studio hoặc công cụ trực quan hóa dữ liệu tương đương.'

In [38]:
benefits = get_benefits(soup)
benefits

'III. Quyền lợi Mức lương: Thỏa thuận xứng đáng theo năng lực và kinh nghiệm thực tế. Cung cấp thiết bị, máy tính làm việc Thưởng: Thưởng định kỳ hoặc thưởng nóng khi hoàn thành tốt các dự án chuyển đổi số và tối ưu vận hành. Môi trường: Làm việc trong môi trường giáo dục năng động, trẻ trung; đồng nghiệp thân thiện, hỗ trợ nhiệt tình. Phát triển: Lộ trình thăng tiến rõ ràng, cơ hội trở thành nhân sự nòng cốt trong chiến lược số hóa của công ty. Đào tạo: Được tạo điều kiện tham gia các khóa đào tạo chuyên sâu để cập nhật các công nghệ mới nhất (AI, Big Data...). Đãi ngộ: Chế độ du lịch, nghỉ dưỡng, Party định kỳ; hưởng đầy đủ chính sách BHXH, BHYT theo quy định của Nhà nước.'

## Nhóm 2: Cty VIP
Nhóm này HOÀN TOÀN KO CÓ `company_tag = soup.find('h1', class_="title")`


- `EDUPIA` https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-2-nam-kinh-nghiem-j2094369.html
- `VPBANK` https://www.topcv.vn/brand/vpbank/tuyen-dung/senior-data-analytics-debt-collection-banking-ha-noi-ta150-j2120972.html

In [40]:
# Link web đặc biệt
# https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-2-nam-kinh-nghiem-j2094369.html

job_link = 'https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-2-nam-kinh-nghiem-j2094369.html'
job_link = job_link.split('?')[0]



# Mở trang job chi tiết
driver.get(job_link)
time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn

# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height



# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

soup = new_soup

### 2.1. company_name

In [46]:

# 1. Tìm thẻ <h1> "title" duy nhất chứa tên công ty
company_tag = soup.find('h1', class_="title")
if not company_tag:
    print("ko có")

ko có


In [45]:
# 1. Tìm thẻ <h1> "title" duy nhất chứa tên công ty
company_tag = soup.find('h1', class_="title")
if company_tag:
    print("có")

In [47]:
# 2. Tìm thẻ chứa tên cty, cũng là tag có tính chất duy nhất
company_tag = soup.find('div', class_="footer-info-content footer-info-company-name")
company_tag

<div class="footer-info-content footer-info-company-name">EDUPIA</div>

In [48]:
company_name = company_tag.get_text(strip=True)
company_name

'EDUPIA'

In [49]:
# function 6: lấy tên cty

def get_company(soup):
    try:
        company_tag = soup.find('div', class_="footer-info-content footer-info-company-name")
        
        if company_tag:
            return company_tag.get_text(strip=True)

        return None
    except Exception:
        return None


company_name = get_company(soup)
company_name

'EDUPIA'

### 2.2. Title

In [52]:
# lấy thông tin bên box bên trái, chứa tên job
header_box = soup.find_all('div', class_="box-header")
header_box

[<div class="box-header">
 <h2 class="title">
                                                                     Data Governance Specialist - 2 Năm Kinh Nghiệm
                                                                                                 <span class="icon-verified-employer level-five">
 <i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
   &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã được duyệt Giấy phép kinh doanh&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt

In [53]:
len(header_box)

1

In [63]:
# lấy thông tin bên box bên trái, chứa tên job, chỉ có 1 tag duy nhất
header_box = soup.find('div', class_="box-header")
header_box

<div class="box-header">
<h2 class="title">
                                                                    Data Governance Specialist - 2 Năm Kinh Nghiệm
                                                                                                <span class="icon-verified-employer level-five">
<i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
  &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã được duyệt Giấy phép kinh doanh&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt

In [56]:
# lấy title_tag
title_tag = header_box.find_all('h2', class_='title')
title_tag

[<h2 class="title">
                                                                     Data Governance Specialist - 2 Năm Kinh Nghiệm
                                                                                                 <span class="icon-verified-employer level-five">
 <i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
   &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã được duyệt Giấy phép kinh doanh&lt;/span&gt;&lt;br&gt;
   &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Tài khoản NTD 

In [57]:
len(title_tag)

1

In [58]:
# lấy title_tag
title_tag = header_box.find('h2', class_='title')
title_tag

<h2 class="title">
                                                                    Data Governance Specialist - 2 Năm Kinh Nghiệm
                                                                                                <span class="icon-verified-employer level-five">
<i class="fa-solid fa-circle-check icon-verified-employer-tooltip" data-container="body" data-html="true" data-original-title="
  &lt;b&gt;Nhà tuyển dụng&lt;/b&gt;&lt;span&gt; đã được xác thực:&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực email tên miền công ty&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã xác thực số điện thoại&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Đã được duyệt Giấy phép kinh doanh&lt;/span&gt;&lt;br&gt;
  &lt;span&gt;&lt;i class='fa-solid fa-circle-check color-green'&gt;&lt;/i&gt; Tài khoản NTD được tạo 

In [59]:
title = title_tag.get_text(separator=' ', strip=True)
title

'Data Governance Specialist - 2 Năm Kinh Nghiệm'

In [60]:
# function 1: lấy tiêu đề job

def get_title(soup):
    try:
        # lấy thông tin bên box bên trái, chứa tên job
        header_box = soup.find('div', class_="box-header")
        
        if header_box:
            title_tag = header_box.find('h2', class_='title')

            
            if title_tag:
                # Cách an toàn nhất: Lấy toàn bộ text, sau đó xóa các khoảng trắng thừa
                raw_title = title_tag.get_text(separator=' ', strip=True)

                
                return raw_title
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy title: {e}")
        return None


# test function: lấy tiêu đề job
title = get_title(new_soup)
title

'Data Governance Specialist - 2 Năm Kinh Nghiệm'

### 2.3. job_deadline

In [66]:
deadline_tag = header_box.find('span', class_="deadline")
deadline_tag

<span class="deadline">Còn
                                            <strong>1</strong>
                                            ngày để ứng tuyển</span>

In [68]:
if deadline_tag:
    # Cách an toàn nhất: Lấy toàn bộ text, sau đó xóa các khoảng trắng thừa
    raw_deadline = deadline_tag.get_text(separator=' ', strip=True)

raw_deadline   

'Còn 1 ngày để ứng tuyển'

In [ ]:

def get_deadline_date(header_box):
    try:
        deadline_tag = header_box.find('span', class_="deadline")

        if deadline_tag:
            # Cách an toàn nhất: Lấy toàn bộ text, sau đó xóa các khoảng trắng thừa
            raw_text = deadline_tag.get_text(separator=' ', strip=True)

            now = datetime.datetime.now() # Lấy thời điểm hiện tại lúc crawl



            # 1. Nếu hết hạn trong vài giờ, nghĩa là hạn chót vẫn là ngày hôm nay
            if 'hôm nay' in raw_text or 'giờ' in raw_text or 'phút' in raw_text or 'giây' in raw_text:
                deadline_date = now


            else:
                # 2. Dùng Regex để tách lấy con số trong chuỗi
                match = re.search(r'\d+', raw_text)
                if not match:
                    return None
                
                number = int(match.group())
                

                # 2. Logic tính toán dựa trên đơn vị thời gian
                if 'ngày' in raw_text:
                    # Cộng thêm số ngày tương ứng
                    deadline_date = now + datetime.timedelta(days=number)
                elif 'tuần' in raw_text:
                    # Quy đổi tạm thời 1 tuần = 7 tuần
                    deadline_date = now + datetime.timedelta(days=number * 7)            
                elif 'tháng' in raw_text:
                    # Quy đổi tạm thời 1 tháng = 30 ngày
                    deadline_date = now + datetime.timedelta(days=number * 30)
                else:
                    # Các trường hợp khác không xác định
                    return None
                

        # 3. Trả về định dạng ngày chuẩn YYYY-MM-DD để sau này dễ vẽ biểu đồ
        return deadline_date.strftime("%d/%m/%Y")

    except (AttributeError, IndexError, Exception):
        # Nếu không tìm thấy thẻ hoặc có lỗi tính toán, trả về None
        return None

In [79]:
job_deadline = get_deadline_date(header_box)
job_deadline

'24/04/2026'

### 2.4.      salary
### 2.5.      work_type
### 2.6.      education_level
### 2.7.      hire_number
### 2.8.      level_employee
### 2.9.      years_experience

In [84]:
# Lấy title của box chứa thông tin. Cái title tra kiểu này chỉ cho 1 kết quả

info_title_tag = soup.find('h2', class_="title", string="Thông tin")
info_title_tag

<h2 class="title">Thông tin</h2>

In [85]:
info_box = info_title_tag.parent
info_box

<div class="box-info">
<h2 class="title">Thông tin</h2>
<div class="box-main">
<div class="box-item">
<div class="box-img">
<i class="fa-solid fa-money-bill-wave"></i>
</div>
<div>
<strong>Mức lương</strong> <br/>
<span>
                                                Tới 40 triệu
                                            </span>
</div>
</div>
<div class="box-item">
<div class="box-img">
<i class="fa-solid fa-user-group"></i>
</div>
<div>
<strong>Số lượng tuyển </strong> <br/>
<span>1 người</span>
</div>
</div>
<div class="box-item">
<div class="box-img">
<i class="fa-solid fa-calendar-clock"></i>
</div>
<div>
<strong>Hình thức làm việc </strong> <br/>
<span>Toàn thời gian</span>
</div>
</div>
<div class="box-item">
<div class="box-img">
<i class="fa-solid fa-chart-user"></i>
</div>
<div>
<strong>Cấp bậc </strong> <br/>
<span>Nhân viên</span>
</div>
</div>
<div class="box-item">
<div class="box-img">
<svg height="22" viewbox="0 0 17 18" width="22" xmlns="http://www.w3.org/2000/svg">


In [87]:
group_box = info_box.find_all('div', class_='box-item')
group_box

[<div class="box-item">
 <div class="box-img">
 <i class="fa-solid fa-money-bill-wave"></i>
 </div>
 <div>
 <strong>Mức lương</strong> <br/>
 <span>
                                                 Tới 40 triệu
                                             </span>
 </div>
 </div>,
 <div class="box-item">
 <div class="box-img">
 <i class="fa-solid fa-user-group"></i>
 </div>
 <div>
 <strong>Số lượng tuyển </strong> <br/>
 <span>1 người</span>
 </div>
 </div>,
 <div class="box-item">
 <div class="box-img">
 <i class="fa-solid fa-calendar-clock"></i>
 </div>
 <div>
 <strong>Hình thức làm việc </strong> <br/>
 <span>Toàn thời gian</span>
 </div>
 </div>,
 <div class="box-item">
 <div class="box-img">
 <i class="fa-solid fa-chart-user"></i>
 </div>
 <div>
 <strong>Cấp bậc </strong> <br/>
 <span>Nhân viên</span>
 </div>
 </div>,
 <div class="box-item">
 <div class="box-img">
 <svg height="22" viewbox="0 0 17 18" width="22" xmlns="http://www.w3.org/2000/svg">
 <path d="M11.8812 11.548C12.3502 

In [88]:
len(group_box)

6

In [89]:
job_general_info = {}

for block in group_box:
    lable_tag = block.find('strong')
    value_tag = block.find('span')
    
    # Chỉ lưu nếu có đủ cả title và value
    if lable_tag and value_tag:
        lable_text = lable_tag.get_text(separator=" ", strip=True)
        value_text = value_tag.get_text(separator=" ", strip=True)
        job_general_info[lable_text] = value_text


job_general_info        

{'Mức lương': 'Tới 40 triệu',
 'Số lượng tuyển': '1 người',
 'Hình thức làm việc': 'Toàn thời gian',
 'Cấp bậc': 'Nhân viên',
 'Học vấn': 'Đại Học trở lên',
 'Kinh nghiệm': '2 năm'}

In [91]:
# Dictionary 6 phần: LƯƠNG + WORK_TYPE + CẤP BẬC + HỌC VẤN + HIRE_NUMBER + TRÌNH ĐỘ NHÂN VIÊN + NĂM KINH NGHIỆM 


def job_general_info(soup):
    """
    Quét HTML 1 lần, gom toàn bộ thông tin thành Dictionary
    """
    job_general_info = {}
    
    try:
        # 1. Lấy title của box chứa thông tin. Cái title tra kiểu này chỉ cho 1 kết quả
        info_title_tag = soup.find('h2', class_="title", string="Thông tin")
        info_title_tag


        # 2. Lấy thẻ cha của info_title_tag
        info_box = info_title_tag.parent



        if info_box:
            # Tìm tất cả các khối nhỏ chứa thông tin (label + value)
            group_box = info_box.find_all('div', class_='box-item')
            
            
            for block in group_box:
                lable_tag = block.find('strong')
                value_tag = block.find('span')
                
                # Chỉ lưu nếu có đủ cả title và value
                if lable_tag and value_tag:
                    lable_text = lable_tag.get_text(separator=" ", strip=True)
                    value_text = value_tag.get_text(separator=" ", strip=True)
                    job_general_info[lable_text] = value_text
                                
        return job_general_info

            
    except Exception as e:
        logging.error(f"Lỗi khi lấy job_general_info: {e}")
        return job_head_info # Trả về dict rỗng nếu lỗi

# ================= ÁP DỤNG =================

# Gọi hàm 1 lần duy nhất
job_general_info = job_general_info(soup)
job_general_info


{'Mức lương': 'Tới 40 triệu',
 'Số lượng tuyển': '1 người',
 'Hình thức làm việc': 'Toàn thời gian',
 'Cấp bậc': 'Nhân viên',
 'Học vấn': 'Đại Học trở lên',
 'Kinh nghiệm': '2 năm'}

In [92]:
# Lấy từng giá trị ra cực kỳ nhanh và gọn bằng hàm .get() của Dict
# Nếu ko có, thì .get() trả về NONE
salary              = job_general_info.get('Mức lương')
hire_number         = job_general_info.get('Số lượng tuyển')
work_type           = job_general_info.get('Hình thức làm việc')
level_employee      = job_general_info.get('Cấp bậc')
education_level     = job_general_info.get('Học vấn')
years_experience    = job_general_info.get('Kinh nghiệm')

In [93]:
salary

'Tới 40 triệu'

In [94]:
hire_number

'1 người'

In [95]:
work_type

'Toàn thời gian'

In [96]:
level_employee

'Nhân viên'

In [97]:
education_level

'Đại Học trở lên'

In [98]:
years_experience

'2 năm'

### 2.10. expertise
Chuyên môn của nhân viên (Data Analyst Chăm sóc sức khỏe / Làm đẹp Y tế / Dược phẩm)

In [105]:
# 1. Tìm expertise_tag  "job-tags__group-name"
pattern_class = re.compile(r'(?=.*job-tags)(?=.*group-name)', re.IGNORECASE)


expertise_tag = soup.find(
    'div', 
    class_=pattern_class, 
    string=re.compile(r'Chuyên môn', re.IGNORECASE))

expertise_tag

<div class="job-tags__group-name">
                    Chuyên môn:
                </div>

In [106]:
if expertise_tag:
    # 2. Lấy thẻ cha của nó
    parent_expertise = expertise_tag.parent 

parent_expertise

<div class="job-tags__group">
<div class="job-tags__group-name">
                    Chuyên môn:
                </div>
<div class="job-tags__group-list-tag">
<div class="job-tags__group-list-tag-scroll">
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-data-analyst-cr257cb261cl145" target="_blank" title="Chuyên môn Data Analyst">
                            Chuyên môn Data Analyst
                        </a>
</div>
</div>
</div>

In [107]:
# Tìm thẻ div anh em tiếp theo có class là 'job-tags__group-list-tag'
expertise_list = parent_expertise.find('div', class_=re.compile(r'group-list-tag-scroll'))
expertise_list

<div class="job-tags__group-list-tag-scroll">
<a class="item search-from-tag link" href="https://www.topcv.vn/tim-viec-lam-data-analyst-cr257cb261cl145" target="_blank" title="Chuyên môn Data Analyst">
                            Chuyên môn Data Analyst
                        </a>
</div>

In [108]:

if expertise_list:
    # Lấy tất cả thẻ <a> bên trong
    a_tags = expertise_list.find_all('a')
    
    # Dùng list comprehension để lấy text, bỏ khoảng trắng thừa
    tags_list = [a.get_text(separator=" ", strip=True) for a in a_tags if a.text]
    
    # Nối thành chuỗi cách nhau bởi dấu phẩy
    if tags_list:
        expertise = ", ".join(tags_list)


expertise

'Chuyên môn Data Analyst'

In [ ]:
# function 9: Lấy Chuyên môn nghiệp vụ của ứng viên


def get_job_expertise(soup):
    """Lấy danh sách các thẻ chuyên môn (Expertise / Tags) của công việc."""
    try:
        # 1. Tìm thẻ label chứa chữ "Chuyên môn"
        pattern_class = re.compile(r'(?=.*job-tags)(?=.*group-name)', re.IGNORECASE)
        pattern_string=re.compile(r'Chuyên môn', re.IGNORECASE)

        expertise_tag = soup.find('div', class_=pattern_class, string=pattern_string)
        
        # 2. Phải lồng các bước tiếp theo vào TRONG lệnh if này để đảm bảo an toàn
        if expertise_tag:
            parent_expertise = expertise_tag.parent 
            
            if parent_expertise:
                # Tìm khối cuộn chứa các thẻ tag
                expertise_list = parent_expertise.find('div', class_=re.compile(r'group-list-tag-scroll'))
                
                if expertise_list:
                    # Lấy tất cả thẻ <a> bên trong
                    a_tags = expertise_list.find_all('a')
                    
                    # Dùng list comprehension để lấy text, loại bỏ các thẻ rỗng
                    tags_list = [a.get_text(strip=True) for a in a_tags if a.get_text(strip=True)]
                    
                    # Nối thành chuỗi cách nhau bởi dấu phẩy, và xóa hết các text "chuyên môn"
                    if tags_list:
                        return ", ".join(tags_list).replace('Chuyên môn', '').strip()
                            
        # Trả về None nếu trang web thiếu một trong các khối HTML trên
        return None 
        
    except Exception as e:
        # Log lỗi giống hệt các function khác trong file của bạn
        logging.error(f"Lỗi khi lấy job_expertise: {e}")
        return None


In [112]:
expertise_employee = get_job_expertise(new_soup)
expertise_employee

'Data Analyst'

### 2.11. Location

In [130]:
# Có box là footer-info
box_footer = soup.find('div', class_='footer-info')
box_footer 

<div class="footer-info">
<div class="footer-info-content footer-info-company-name">EDUPIA</div>
<div class="footer-info-title">Địa chỉ:</div>
<div class="footer-info-content">Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội</div>
<div class="footer-info-title">Tel:</div>
<div class="footer-info-content">0853947232</div>
<div class="footer-info-title">Website:</div>
<div class="footer-info-content"><a href="https://edupia.vn/" target="_blank">https://edupia.vn/</a></div>
</div>

In [131]:
# Bước 1: Tìm đúng thẻ div có class là 'footer-info-title' VÀ chứa text là 'Địa chỉ:'
address_label = box_footer.find('div', class_='footer-info-title', string='Địa chỉ:')
address_label

<div class="footer-info-title">Địa chỉ:</div>

In [ ]:
if address_label:
    # Bước 2: Lấy thẻ div "anh em" ngay sát bên dưới nó (cùng cấp) có class 'footer-info-content'
    address_content = address_label.find_next_sibling('div', class_='footer-info-content')
    
    if address_content:
        # Lấy phần text bên trong và loại bỏ khoảng trắng thừa
        address_text = address_content.text.strip()

address_text

'Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội'

In [138]:
# DANH SÁCH CHUẨN 34 TỈNH THÀNH (Sau sáp nhập 2025)
# Key: chữ thường để search | Value: Tên chuẩn để lưu vào Data
PROVINCES = {
    "tuyên quang": "Tuyên Quang",
    "lào cai": "Lào Cai",
    "thái nguyên": "Thái Nguyên",
    "phú thọ": "Phú Thọ",
    "bắc ninh": "Bắc Ninh",
    "hưng yên": "Hưng Yên",
    "hải phòng": "Hải Phòng",
    "ninh bình": "Ninh Bình",
    "quảng trị": "Quảng Trị",
    "đà nẵng": "Đà Nẵng",
    "quảng ngãi": "Quảng Ngãi",
    "gia lai": "Gia Lai",
    "khánh hòa": "Khánh Hòa",
    "điện biên": "Điện Biên",
    "hà nội": "Hà Nội",
    "hà tĩnh": "Hà Tĩnh",
    "lạng sơn": "Lạng Sơn",
    "lai châu": "Lai Châu",
    "nghệ an": "Nghệ An",
    "quảng ninh": "Quảng Ninh",
    "sơn la": "Sơn La",
    "thanh hóa": "Thanh Hóa",
    "cao bằng": "Cao Bằng",
    "huế": "Huế",
    "lâm đồng": "Lâm Đồng",
    "đắk lắk": "Đắk Lắk",
    "hồ chí minh": "Hồ Chí Minh",
    "đồng nai": "Đồng Nai",
    "tây ninh": "Tây Ninh",
    "cần thơ": "Cần Thơ",
    "vĩnh long": "Vĩnh Long",
    "đồng tháp": "Đồng Tháp",
    "cà mau": "Cà Mau",
    "an giang": "An Giang"
}

def extract_new_location(address_text):
    if not address_text:
        return None
        
    # Đưa tất cả về chữ thường
    text = address_text.lower()
    text = text.replace("việt nam", "").replace("vietnam", "")
    
    # Tách chuỗi theo dấu phẩy hoặc chấm
    parts = re.split(r'[,|.]', text)
    
    # Duyệt ngược từ cuối lên
    for part in reversed(parts):
        clean_part = part.strip()
        # Xóa các chữ "tỉnh", "thành phố", "tp.", "tp" ở đầu mỗi cụm
        clean_part = re.sub(r'^(thành phố|tp\.|tp|tỉnh)\s+', '', clean_part).strip()
        
        # Chỉ check đúng 34 tỉnh thành trong list
        if clean_part in PROVINCES:
            return PROVINCES[clean_part] 
            
    return None



location = extract_new_location(address_text)
location

'Hà Nội'

In [140]:
# Function lấy thông tin địa chỉ của job

def get_location(soup):
    # Có box là footer-info
    box_footer = soup.find('div', class_='footer-info')

    # Bước 1: Tìm đúng thẻ div có class là 'footer-info-title' VÀ chứa text là 'Địa chỉ:'
    address_label = box_footer.find('div', class_='footer-info-title', string='Địa chỉ:')

    if address_label:
        # Bước 2: Lấy thẻ div "anh em" ngay sát bên dưới nó (cùng cấp) có class 'footer-info-content'
        address_content = address_label.find_next_sibling('div', class_='footer-info-content')
    
        if address_content:
            # Lấy phần text bên trong và loại bỏ khoảng trắng thừa
            address_text = address_content.text.strip()

            if address_text:
                return extract_new_location(address_text)    

    return None    



location = get_location(soup)
location

'Hà Nội'

### 2.12      MÔ TẢ CÔNG VIỆC
### 2.13      YÊU CẦU
### 2.14      PHÚC LỢI

In [113]:
# Lấy trước tag title Mô tả công việc
description_tag = soup.find('h2', class_='title', string='Mô tả công việc')
description_tag

<h2 class="title">Mô tả công việc</h2>

In [114]:
if description_tag:
    # 2. Lấy thẻ cha của nó
    parent_description = description_tag.parent 

parent_description

<div class="box-info">
<h2 class="title">Mô tả công việc</h2>
<div class="content-tab"><p>− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. <br/>− Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata...) và xây dựng các chỉ số đo lường chất lượng dữ liệu. <br/>− Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI. <br/>− Hỗ trợ tuân thủ các quy định về an toàn thông tin, bảo mật dữ liệu, phân quyền truy cập dữ liệu, đảm bảo luật định (Luật Dữ liệu, Luật Bảo vệ Dữ liệu cá nhân, ...). <br/>− Phối hợp với các đơn vị nghiệp vụ, sản phẩm, IT và DE/BI để đảm bảo dữ liệu được quản trị đúng chuẩn và nhất quán. <br/>− Tham gia xây dựng tài liệu, hướng dẫn và đào tạo về Data Governance cho các bên liên quan. </p></div>
</div>

In [115]:
# BOX nội dung chính
description_content_tag = parent_description.find('div', class_='content-tab')
description_content_tag

<div class="content-tab"><p>− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. <br/>− Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata...) và xây dựng các chỉ số đo lường chất lượng dữ liệu. <br/>− Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI. <br/>− Hỗ trợ tuân thủ các quy định về an toàn thông tin, bảo mật dữ liệu, phân quyền truy cập dữ liệu, đảm bảo luật định (Luật Dữ liệu, Luật Bảo vệ Dữ liệu cá nhân, ...). <br/>− Phối hợp với các đơn vị nghiệp vụ, sản phẩm, IT và DE/BI để đảm bảo dữ liệu được quản trị đúng chuẩn và nhất quán. <br/>− Tham gia xây dựng tài liệu, hướng dẫn và đào tạo về Data Governance cho các bên liên quan. </p></div>

In [119]:
if description_content_tag:
    # Lấy text, dùng dấu cách ' ' để các đoạn <p> không bị dính liền vào nhau
    raw_text = description_content_tag.get_text(separator=' ', strip=True)
    
    # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
    clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)
    
    # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
    final_text = ' '.join(clean_text.split())
    
    # Chống lỗi công thức khi xuất ra file CSV/Excel
    if final_text.startswith(('-', '=', '+', '@')):
        final_text = "'" + final_text


final_text

'Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata...) và xây dựng các chỉ số đo lường chất lượng dữ liệu. Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI. Hỗ trợ tuân thủ các quy định về an toàn thông tin, bảo mật dữ liệu, phân quyền truy cập dữ liệu, đảm bảo luật định (Luật Dữ liệu, Luật Bảo vệ Dữ liệu cá nhân, ...). Phối hợp với các đơn vị nghiệp vụ, sản phẩm, IT và DE/BI để đảm bảo dữ liệu được quản trị đúng chuẩn và nhất quán. Tham gia xây dựng tài liệu, hướng dẫn và đào tạo về Data Governance cho các bên liên quan.'

In [121]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def extract_paragraph_content(soup, heading_keyword):
    """
    Hàm đa năng: Tìm thẻ h2 chứa keyword, sau đó lấy toàn bộ text 
    của các thẻ anh em (siblings) nằm dưới nó trong cùng 1 div.
    """
    try:
        # 1. thẻ title của box:
        pattern_string = re.compile(rf'{heading_keyword}', re.IGNORECASE)
        h2_tag = soup.find('h2', class_='title', string=pattern_string)


        # 2. Tìm thẻ cha là BOX chứa cái thông tin đó
        if h2_tag:
            box_tag = h2_tag.parent 

            # 3. BOX nội dung chính
            if box_tag:
                content_tag = box_tag.find('div', class_='content-tab')

                # 4. Lấy nguyên text
                if content_tag:
                    # Lấy text, dùng dấu cách ' ' để các đoạn <p> không bị dính liền vào nhau
                    raw_text = content_tag.get_text(separator=' ', strip=True)
                    
                    # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
                    clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)
                    
                    # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
                    final_text = ' '.join(clean_text.split())
                    
                    # Chống lỗi công thức khi xuất ra file CSV/Excel
                    if final_text.startswith(('-', '=', '+', '@')):
                        final_text = "'" + final_text
                
                    return final_text
            
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {heading_keyword}: {e}")
        return None



# ==========================================
# ÁP DỤNG VÀO 3 HÀM CỦA BẠN
# ==========================================

def get_job_description(soup):
    return extract_paragraph_content(soup, "Mô tả công việc")

def get_job_requirements(soup):
    return extract_paragraph_content(soup, "Yêu cầu ứng viên")

def get_benefits(soup):
    return extract_paragraph_content(soup, "Quyền lợi được hưởng")

In [122]:
description = get_job_description(soup)
description

'Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata...) và xây dựng các chỉ số đo lường chất lượng dữ liệu. Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI. Hỗ trợ tuân thủ các quy định về an toàn thông tin, bảo mật dữ liệu, phân quyền truy cập dữ liệu, đảm bảo luật định (Luật Dữ liệu, Luật Bảo vệ Dữ liệu cá nhân, ...). Phối hợp với các đơn vị nghiệp vụ, sản phẩm, IT và DE/BI để đảm bảo dữ liệu được quản trị đúng chuẩn và nhất quán. Tham gia xây dựng tài liệu, hướng dẫn và đào tạo về Data Governance cho các bên liên quan.'

In [123]:
requirements = get_job_requirements(soup)
requirements

'Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ Data Management. Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế. Có tư duy hệ thống, kỹ năng làm việc với nhiều bên liên quan. Ưu tiên ứng viên có kinh nghiệm làm việc triển khai các dự án quản trị dữ liệu, thiết kế hạ tầng dữ liệu và phân quyền.'

In [124]:
benefits = get_benefits(soup)
benefits

'Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh bạch, dựa trên năng lực và đóng góp thực tế. Tham gia đầy đủ BHXH, BHYT theo Luật Lao động hiện hành, cùng 12 ngày phép năm và các chế độ nghỉ lễ theo quy định. Được nhận Bộ sản phẩm học tiếng Anh online trọn đời của công ty (trị giá 7.200.000 VNĐ) ngay khi nhận việc. Được làm việc trong văn phòng trẻ trung, không gian mở, khơi nguồn cảm hứng và khuyến khích sáng tạo. Được tham gia nhiều hoạt động gắn kết: Tham gia du lịch nghỉ dưỡng hàng năm tại các điểm đến hấp dẫn, cùng các hoạt động Team Building và gắn kết văn hóa nội bộ được tổ chức thường xuyên.'